# Olentangy Data Science Day 2026

*A hands-on introduction to simulation, modeling and problem-solving.*

<br/>

<img src="images/bread_financial_logo.svg" width="200" />

## About Bread Financial

<img src="images/Screenshot Bread Financial General 01.png" width="100%" />

<img src="images/Screenshot Bread Financial General 02.png" width="100%" />

## About Me

Principal data scientist at **Bread Financial**.

Today I'll walk you through some fun experiments that show how data science works in practice.

## Today's Demo

- Explore classic probability puzzles through **simulation and visualization**
- Understand *tokens* and *embeddings* used in *LLMs*
- Design a marketing campaign for a local coffee shop

<br/>

> Data scientists use similar approaches every day to test ideas, validate assumptions, and discover insights.

<br/>

All code and notebooks: **[github.com/wl37/datascienceday2026](https://github.com/wl37/datascienceday2026)**

## Part 1: Simulation (Probability and Statistics)

### The Coin Flip — How random is random?

Let's simulate thousands of coin flips and see what the data tells us.

In [ ]:
import random
import matplotlib.pyplot as plt

def simulate_coin_flips(n: int) -> tuple[float, float]:
    flips = [random.choice(['H', 'T']) for _ in range(n)]
    heads = flips.count('H') / n * 100
    tails = flips.count('T') / n * 100
    return heads, tails

for n in [10, 100, 1000, 10_000]:
    h, t = simulate_coin_flips(n)
    print(f"{n:>6} flips → Heads: {h:.1f}%  Tails: {t:.1f}%")

### 🎂 Birthday Match

<img src="images/birthday_match.png" width="400" />

Raise your hand when your birthday falls in the group — and let's see if we can find a match!


| Round | Months |
|:-----:|:-------|
| 1 | **January – March** |
| 2 | **April – June** |
| 3 | **July – September** |
| 4 | **October – December** |


#### Re-frame the Same Question

Do you think a shared birthday among a room of people like this is,  

1.rare; 2. possible, or 3. actually likely?

OR, 
For a group of 23 people, would you guess the chance of a match is:

- below 25%?
- around 50%?
- above 75%?

#### Let's simulate!

We'll get back to the math — but there's a way to **touch & feel** it yourself. With AI and coding assistant, everyone can design and run simulations. How would you design a simulation for this problem?


What if we just **built the room** inside the computer? Give 23 people each a random birthday, then check whether any two match. Do that thousands of times and count how often a match happens.

In [ ]:
import random

def has_birthday_match(group_size: int) -> bool:
    """Simulate a room of people with random birthdays; return True if any two share one."""
    birthdays = [random.randint(1, 365) for _ in range(group_size)]
    return len(birthdays) != len(set(birthdays))


def estimate_match_probability(group_size: int, trials: int = 10_000) -> float:
    """Run the experiment many times and return the fraction that had a match."""
    matches = sum(has_birthday_match(group_size) for _ in range(trials))
    return matches / trials


for n in [10, 15, 20, 23, 30, 40, 50]:
    prob = estimate_match_probability(n)
    print(f"{n:>2} people  →  {prob * 100:5.1f}% chance of a shared birthday")

#### Why Is It So High?

The simulation gave us the answer — but *why* does a group of just 23 cross 50%? Let's look at the math behind what the computer just did.

**What is the probability that nobody matches?**

- The first person can have any birthday: $\frac{365}{365}$
- The second person must avoid that one day: $\frac{364}{365}$
- The third person must avoid two taken days: $\frac{363}{365}$
- And the pattern continues

So for a group of size $n$:

$$P(\text{no match}) = \frac{365}{365} \times \frac{364}{365} \times \frac{363}{365} \times \cdots \times \frac{365-n+1}{365}$$

But the original question asks for **at least one match**, so we flip the answer:

$$P(\text{at least one match}) = 1 - P(\text{no match})$$

For $n = 23$:

$$P(\text{at least one match}) \approx 0.507$$

That is just over **50%**.

#### Reflect

Why does it rise so quickly? Because the number of possible pairs grows fast:

$$\binom{n}{2} = \frac{n(n-1)}{2}$$

For 23 people, that is **253 pairs**. The room does not feel very large, but the number of chances for a match is already much bigger than most people expect.

### The Monty Hall Show

It's the 1970s. You're a contestant on *Let's Make a Deal*. The host, Monty Hall, stands in front of three closed doors.

Behind one door is a brand-new car. Behind the other two — goats.

You pick a door. Say, Door #1. But before you open it, Monty — who knows *exactly* what's behind every door — reveal another door that has a goat. 

Now only two doors remain: a car and a goat. Monty would give you one more chance:

> ***"Would you like to switch?"***

#### What Would You Do?

Two doors left. One car. Before we go any further, commit to an answer:

- **Stay** with your original door?
- **Switch** to the other one?
- **Doesn't matter** — it's the same either way?

Lock in your choice. Don't change it once you see what comes next.

#### Let's Play a Thousand Times - Simulation

Arguments won't settle this — but data will. How would you design a simulation for this game?


Think about what one round looks like:
1. Randomly place the car behind one of three doors.
2. The player picks a door.
3. Monty opens a door that is **not** the player's pick and **not** the car.
4. The player either stays or switches.
5. Record whether they won.

Now repeat that thousands of times — once for a player who always stays, and once for a player who always switches. Then compare.

In [ ]:
import random

def run_monty_hall(num_trials: int, switch_door: bool) -> int:
    """Play the Monty Hall game num_trials times and return total wins."""
    wins = 0
    doors = [0, 1, 2]

    for _ in range(num_trials):
        prize_door = random.choice(doors)
        player_pick = random.choice(doors)

        # Monty opens a door that is neither the prize nor the player's pick
        monty_opens = random.choice(
            [d for d in doors if d != prize_door and d != player_pick]
        )

        if switch_door:
            final = [d for d in doors if d != player_pick and d != monty_opens][0]
        else:
            final = player_pick

        if final == prize_door:
            wins += 1

    return wins


In [ ]:
for n in [100, 1_000, 10_000]:
    stay_wins   = run_monty_hall(n, switch_door=False)
    switch_wins = run_monty_hall(n, switch_door=True)
    print(f"{n:>6} games  →  Stay: {stay_wins/n*100:5.1f}%   Switch: {switch_wins/n*100:5.1f}%")

#### So Why Does Switching Work?

When you first picked a door, you had a **1-in-3** chance of being right. That means there was a **2-in-3** chance the car was behind one of the *other* doors.

Now here is the key: Monty is not choosing randomly. He *knows* where the car is and he *always* opens a goat door. He is giving you information. That entire $\frac{2}{3}$ probability collapses onto the single remaining door.

- **Stay** — you win only if your first guess was right — $\frac{1}{3}$
- **Switch** — you win whenever your first guess was *wrong* — $\frac{2}{3}$

Switching doesn't just help a little. It **doubles** your chances.

#### Reflect

The lesson is the same one the birthday problem taught us: when your gut says one thing and the simulation says another. And sometimes the best way to believe the math is to run the experiment yourself.

*Fun fact: Someone in the 90s found out this 'bug' of the show.*

## Part 2: Words in LLMs

### How Does ChatGPT Know What Words Mean?

When you type "dog" into ChatGPT, it doesn't look the word up in a dictionary. It has never *read* a dictionary.

Yet it knows that **"dog"** and **"puppy"** are related. It knows that **"Paris"** is to **"France"** as **"Tokyo"** is to **"Japan"**.

How?

#### Think About It

Computers only understand numbers. So before any AI can work with language, every word has to become a number — or better, a *list* of numbers.

If you had to turn words into numbers so that **similar words get similar numbers**, how would you do it?

- Could you just number them alphabetically? Would "aardvark" and "abandon" really be related?
- What about grouping by topic? How many topics are there?
- What if each word got **hundreds** of numbers — each one capturing a different shade of meaning?

#### Words as Vectors

That last idea is exactly what modern AI does. Every word becomes a **vector** — a list of 300 numbers — learned by reading billions of sentences. Words that appear in similar contexts end up with similar vectors.

```
king   → [0.32, -0.51, 0.78, 0.14, ...300 numbers...]
queen  → [0.30, -0.48, 0.75, 0.19, ...]
dog    → [-0.62, 0.33, -0.10, 0.88, ...]
```

These are called **word embeddings**, and they are the foundation of every modern language model — ChatGPT, Gemini, Claude, all of them.

In [ ]:
from pathlib import Path
import urllib.request
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display
from gensim.models import KeyedVectors
from sklearn.decomposition import PCA

sns.set_theme(style="ticks", font_scale=1.15)
plt.ioff()

data_dir = Path("../data")
zip_path = data_dir / "wiki-news-300d-1M.vec.zip"
vec_path = data_dir / "wiki-news-300d-1M.vec"
url = "https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip"

if not zip_path.exists():
    print("Downloading FastText vectors (first run only, large file)...")
    urllib.request.urlretrieve(url, zip_path)

if not vec_path.exists():
    print("Extracting .vec file...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extract("wiki-news-300d-1M.vec", data_dir)

LOAD_LIMIT = 100_000
ft = KeyedVectors.load_word2vec_format(vec_path, binary=False, limit=LOAD_LIMIT)

print(f"Loaded FastText vectors: {len(ft):,} words, {ft.vector_size} dimensions.")

#### How Do We Measure Similarity?

If every word is a list of 300 numbers, how do we decide whether two words are "close"?

We use **cosine similarity** — it measures how much two vectors point in the same direction:

- **1.0** = identical direction (same meaning)
- **0.0** = perpendicular (unrelated)
- **−1.0** = opposite direction

In [ ]:
def cosine_similarity(v1: np.ndarray, v2: np.ndarray) -> float:
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))


def get_vector(word: str) -> np.ndarray:
    return ft[word]


def nearest_neighbors(word: str, n: int = 5) -> list[tuple[str, float]]:
    return ft.most_similar(word, topn=n)

#### Explore: What Words Live Nearby?

Pick a word from the dropdown and see which words the model considers most similar. Try a few — do the results match your intuition?

In [ ]:
demo_words = [
    "king", "queen", "doctor", "nurse", "dog", "cat", "lion", "python",
    "computer", "internet", "music", "guitar", "happy", "sad", "city", "country",
]

@widgets.interact(
    word=widgets.Dropdown(options=demo_words, value="king", description="Word:",
                          layout=widgets.Layout(width="260px")),
    n=widgets.IntSlider(value=8, min=3, max=12, step=1, description="Show top:",
                        style={"description_width": "initial"},
                        layout=widgets.Layout(width="300px")),
)
def show_neighbors(word, n):
    neighbors = nearest_neighbors(word, n)
    neighbor_words = [w for w, _ in neighbors]
    neighbor_scores = [s for _, s in neighbors]

    fig, ax = plt.subplots(figsize=(8.5, max(2.2, n * 0.4)))
    colors = plt.cm.RdYlGn(np.clip(neighbor_scores, 0, 1))

    bars = ax.barh(range(n), neighbor_scores, color=colors, edgecolor="white", linewidth=1)
    ax.set_yticks(range(n))
    ax.set_yticklabels(neighbor_words, fontsize=11)
    ax.invert_yaxis()
    ax.set_xlim(0.4, 1.0)
    ax.set_xlabel("Cosine Similarity")
    ax.set_title(f'Nearest neighbors for "{word}"', fontsize=14, weight="bold")
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    plt.tight_layout()
    plt.show()

#### Explore: The Word Map

Each word lives in a 300-dimensional space — far too many dimensions for us to picture. But we can use a technique called **PCA** to squash those 300 dimensions down to 2, and draw a map.

If the embeddings are any good, related words should still cluster together even after the projection. Select the groups you want to see and find out.

In [ ]:
word_groups = {
    "royalty":    ["king", "queen", "prince", "princess", "duke", "duchess", "throne", "crown"],
    "countries":  ["france", "japan", "china", "germany", "italy", "canada", "brazil", "india"],
    "capitals":   ["paris", "tokyo", "beijing", "berlin", "rome", "ottawa", "brasilia", "delhi"],
    "technology": ["computer", "internet", "software", "hardware", "python", "java", "database", "algorithm"],
    "animals":    ["dog", "cat", "lion", "tiger", "wolf", "bear", "eagle", "shark"],
    "emotions":   ["happy", "sad", "angry", "calm", "joy", "fear", "love", "stress"],
}

cat_palette = {
    "royalty": "#E8575A", "countries": "#00B4D8", "capitals": "#9B5DE5",
    "technology": "#6BCB77", "animals": "#5B8FB9", "emotions": "#FF6B9D",
}

word_groups = {
    group: [w for w in words if w in ft.key_to_index]
    for group, words in word_groups.items()
}

@widgets.interact(
    cats=widgets.SelectMultiple(
        options=sorted(word_groups.keys()),
        value=tuple(sorted(word_groups.keys())),
        description="Groups:",
        layout=widgets.Layout(height="160px", width="230px"),
    ),
    show_labels=widgets.Checkbox(value=True, description="Show word labels"),
)
def plot_word_map(cats, show_labels):
    selected_words, selected_cats = [], []
    for cat in cats:
        for word in word_groups[cat]:
            selected_words.append(word)
            selected_cats.append(cat)

    if len(selected_words) < 3:
        print("Please select at least one group with 3+ words in vocabulary.")
        return

    vectors = np.vstack([get_vector(w) for w in selected_words])
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(vectors)
    var_exp = pca.explained_variance_ratio_

    fig, ax = plt.subplots(figsize=(11, 8))
    for cat in cats:
        mask = np.array([c == cat for c in selected_cats])
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c=cat_palette[cat], s=120, label=cat.capitalize(),
            edgecolors="white", linewidths=0.9, zorder=3, alpha=0.9,
        )
        if show_labels:
            for i, (x, y) in enumerate(coords[mask]):
                w = np.array(selected_words)[mask][i]
                ax.text(x + 0.02, y, w, fontsize=8.5, color=cat_palette[cat],
                        path_effects=[pe.withStroke(linewidth=2, foreground="white")])

    ax.set_xlabel(f"PC 1 ({var_exp[0]*100:.1f}% variance)", fontsize=11)
    ax.set_ylabel(f"PC 2 ({var_exp[1]*100:.1f}% variance)", fontsize=11)
    ax.set_title("Word Map — 300D → 2D via PCA", fontsize=14, weight="bold")
    ax.legend(title="Group", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=10, title_fontsize=11)
    ax.grid(True, alpha=0.25)
    sns.despine()
    plt.tight_layout()
    plt.show()

#### Explore: Word Arithmetic

Here is where things get really strange. Because words are vectors, you can **do math** with them.

What happens if you take the vector for **"paris"**, subtract **"france"**, and add **"japan"**?

You get something very close to **"tokyo"**.

The model has learned that the *relationship* between a country and its capital is a consistent direction in 300-dimensional space. Try the examples below.

In [ ]:
analogy_candidates = {
    "Capital transfer":   ("paris", "france", "japan"),
    "Country transfer":   ("Japan", "Tokyo", "Delhi"),
    "Gender relation":    ("father", "son", "daughter"),
    "Royal gender":       ("queen", "woman", "man"),
    "Verb morphology":    ("walking", "walk", "swim"),
    "Adjective relation": ("good", "bad", "terrible"),
    "Plural form":        ("people", "person", "child"),
}

analogy_examples = {
    name: triple for name, triple in analogy_candidates.items()
    if all(w in ft.key_to_index for w in triple)
}

options = list(analogy_examples.keys())

@widgets.interact(
    example=widgets.Dropdown(options=options, value=options[0], description="Example:",
                             layout=widgets.Layout(width="320px")),
    topn=widgets.IntSlider(value=8, min=3, max=12, step=1, description="Show top:",
                           style={"description_width": "initial"},
                           layout=widgets.Layout(width="300px")),
)
def word_arithmetic(example, topn):
    a, b, c = analogy_examples[example]
    top = ft.most_similar(positive=[a, c], negative=[b], topn=topn)
    best_word, _ = top[0]

    print(f'"{a}" − "{b}" + "{c}" = ?')
    print(f"Top prediction: {best_word}")

    labels = [w for w, _ in top]
    scores = [s for _, s in top]

    fig, ax = plt.subplots(figsize=(8.5, max(3.2, topn * 0.48)))
    bars = ax.barh(range(topn), scores, color=plt.cm.viridis(scores), edgecolor="white")
    ax.set_yticks(range(topn))
    ax.set_yticklabels(labels, fontsize=11)
    ax.invert_yaxis()
    ax.set_xlim(0.35, 1.0)
    ax.set_xlabel("Cosine Similarity")
    ax.set_title(f'"{a}" − "{b}" + "{c}"', fontsize=13.5, weight="bold")
    ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

#### Reflect

No one told the model that Paris is the capital of France. No one gave it a geography textbook. It learned these relationships by reading billions of sentences and noticing which words appear in similar contexts.

This is the same idea behind every modern language model:

1. **Words are vectors.** Each word becomes a point in high-dimensional space.
2. **Similar meanings → nearby points.** Cosine similarity measures how close.
3. **Relationships become directions.** King − Man + Woman ≈ Queen.

When you chat with an AI, this is what's happening under the hood — every word you type gets turned into a vector, and the model navigates that space to find the right response.